# H-1B Employer Data Hub
This notebook consolidates the year-by-year CSV files in the data folder into a single table and then reshapes that table into a long-format file for analysis.

In [ ]:
import pandas as pd
from pathlib import Path

base_dir = Path.cwd()
data_dir = base_dir / 'data'
output_consolidated = base_dir / 'h1b_employer_datahub_consolidated.csv'
output_unpivoted = base_dir / 'h1b_employer_datahub_unpivoted.csv'

csv_files = sorted(data_dir.glob('*.csv'))
print(f'Found {len(csv_files)} CSV files')

frames = []
for path in csv_files:
    year = int(path.stem.split()[-1])
    df = pd.read_csv(path, low_memory=False)

    rename_map = {
        'Fiscal Year   ': 'Fiscal_Year',
        'Employer (Petitioner) Name': 'Employer',
        'Industry (NAICS) Code': 'NAICS',
        'Tax ID': 'Tax_ID',
        'Petitioner State': 'State',
        'Petitioner City': 'City',
        'Petitioner Zip Code': 'ZIP',
        'New Employment Approval': 'Initial Approval',
        'New Employment Denial': 'Initial Denial',
        'Continuation Approval': 'Continuing Approval',
        'Continuation Denial': 'Continuing Denial',
    }

    df = df.rename(columns=rename_map)
    df['Fiscal_Year'] = year

    keep_cols = [
        'Fiscal_Year', 'Employer', 'NAICS', 'Tax_ID', 'State', 'City', 'ZIP',
        'Initial Approval', 'Initial Denial', 'Continuing Approval', 'Continuing Denial'
    ]
    df = df[[c for c in keep_cols if c in df.columns]]
    frames.append(df)

consolidated = pd.concat(frames, ignore_index=True)
consolidated = consolidated.fillna({'Employer': '', 'State': '', 'City': '', 'ZIP': ''})

for col in ['Initial Approval', 'Initial Denial', 'Continuing Approval', 'Continuing Denial']:
    consolidated[col] = pd.to_numeric(consolidated[col], errors='coerce').fillna(0).astype(int)

# Remove any existing output file first to avoid permission issues in the notebook environment
if output_consolidated.exists():
    output_consolidated.unlink()
if output_unpivoted.exists():
    output_unpivoted.unlink()

consolidated.to_csv(output_consolidated, index=False)
print(f'Saved consolidated CSV to {output_consolidated}')

value_columns = ['Initial Approval', 'Initial Denial', 'Continuing Approval', 'Continuing Denial']
unpivoted = consolidated.melt(
    id_vars=['Fiscal_Year', 'Employer', 'NAICS', 'Tax_ID', 'State', 'City', 'ZIP'],
    value_vars=value_columns,
    var_name='Petition_Type',
    value_name='Count'
)
unpivoted = unpivoted.sort_values(['Fiscal_Year', 'Employer', 'Petition_Type']).reset_index(drop=True)
unpivoted.to_csv(output_unpivoted, index=False)
print(f'Saved unpivoted CSV to {output_unpivoted}')


Found 18 CSV files


PermissionError: [Errno 13] Permission denied: 'c:\\Users\\jbrob\\git\\H-1B_Employer_Data_Hub\\h1b_employer_datahub_consolidated.csv'